In [9]:
import json
import numpy as np
from src.utils import parse_raw, simple_line_plot, fourier_transform_plot, segment_data, segment_plot
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
import os

In [10]:
file_path = "raw_data/protocole_apnee_leo_induct.txt"
metadata_path = "raw_data/protocole_apnee_leo_induct.json"

with open(file_path, 'r') as file:
    file_content = file.readlines()
    
header_json, df = parse_raw(file_content)

if os.path.exists(metadata_path):
    with open(metadata_path, 'r') as meta_file:
        metadata = json.load(meta_file)

df.head()

,timestamp,THORAX,X,Y
0,2025-09-19 10:39:26.873,-4.234314,-0.732500,0.491473
1,2025-09-19 10:39:26.878,-4.260254,-0.731250,0.492713
2,2025-09-19 10:39:26.883,-4.290771,-0.731250,0.494574
3,2025-09-19 10:39:26.888,-4.315186,-0.728750,0.490853
4,2025-09-19 10:39:26.893,-4.345703,-0.729375,0.495194


In [11]:
fig = simple_line_plot(df)
fig

In [12]:
fig = fourier_transform_plot(df)
fig

In [13]:
autocorr_thorax = np.correlate(df['THORAX'], df['THORAX'], mode='full')
lags = np.arange(-len(df['THORAX']) + 1, len(df['THORAX']))

fig = px.line(x=lags, y=autocorr_thorax, title='Autocorrelation of THORAX')
fig.update_xaxes(title_text='Lag')
fig.update_yaxes(title_text='Autocorrelation')
fig

In [14]:
autocorr_X = np.correlate(df['X'], df['X'], mode='full')
lags = np.arange(-len(df['X']) + 1, len(df['X']))

fig = px.line(x=lags, y=autocorr_X, title='Autocorrelation of X')
fig.update_xaxes(title_text='Lag')
fig.update_yaxes(title_text='Autocorrelation')
fig

In [15]:
def energie_moyenne(df, column : str ="X"):
    # Find the closest dip to 0 in autocorr_X (excluding the central peak)
    mid = len(autocorr_X) // 2
    # Search for local minima around the center
    search_range = autocorr_X[mid-500:mid+500]
    dip_idx = np.argmin(search_range)
    closest_dip = dip_idx - 500  # relative to center
    print(closest_dip)
    window_size = 5 * abs(closest_dip)
    energies = df[column].rolling(window=window_size, min_periods=1).apply(lambda x: np.mean(x**2), raw=True)
    return energies

fig = px.line(energie_moyenne(df,"X"))
fig

-40


In [16]:
fig = segment_plot(df, metadata.get("sequences", []))
fig